In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/BMnet/checkpoints_clean_train
!cp -r /content/drive/MyDrive/BodyM_Dev/bodym_dataset /content
!cp /content/drive/MyDrive/BMnet/checkpoints_aug_only/split.json \
   /content/drive/MyDrive/BMnet/checkpoints_clean_train/split.json

In [ ]:
# -*- coding: utf-8 -*-
"""
BMnet Training - RUN 2 (augmentation only, NO z-score)
=======================================================
Run 1'den tek fark: target normalization YOK, eski /100 olcekleme geri geldi.
Amac: Run 1'deki gerilemenin (9.43 -> 10.23) z-score'dan mi geldigini izole etmek.

Run 1'le ayni kalanlar:
  - On-the-fly augmentation (ayni parametreler)
  - Binarize + NEAREST resize
  - split.json mekanizmasi (Run 1'in split'ini KOPYALA, asagiya bak)
  - Drive checkpoint/resume mekanizmasi

CALISTIRMADAN ONCE (bir kere):
  !cp /content/drive/MyDrive/BMnet/checkpoints_aug_v2/split.json \
     /content/drive/MyDrive/BMnet/checkpoints_aug_only/split.json
  (klasor yoksa once script'i bir kere import et / Cell 2'yi calistir,
   os.makedirs klasoru olusturur, sonra kopyala, sonra train() baslat)

Beklenen okuma:
  TestA ~9.4 veya alti -> z-score sucluydu
  TestA ~10.2         -> augmentation bu haliyle zarar veriyor
  Arasi               -> ikisi de kismen etkili
"""

# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/BodyM_Dev/bodym_dataset /content

# %% --- CELL 1: Setup & Imports ---

import os
import json
import random
import time
import numpy as np
import pandas as pd
from pathlib import Path

import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


# %% --- CELL 2: Paths & Config ---

DRIVE_BASE = "/content/drive/MyDrive/BMnet"
DATA_ROOT = "/content/bodym_dataset/bodym"
# DEGISTI: yeni deney = yeni klasor (Run 1'in ustune yazma!)
CHECKPOINT_DIR = os.path.join(DRIVE_BASE, "checkpoints_aug_only")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

MEASUREMENT_NAMES = [
    "ankle", "arm-length", "bicep", "calf", "chest", "forearm",
    "height", "hip", "leg-length", "shoulder-breadth",
    "shoulder-to-crotch", "thigh", "waist", "wrist"
]

CONFIG = {
    "img_height": 640,
    "img_width": 480,
    "batch_size": 32,
    "num_workers": 4,
    "lr": 1e-3,
    "total_iterations": 145_000,
    "lr_gamma": 0.1,
    "val_split": 0.1,
    "num_measurements": len(MEASUREMENT_NAMES),
    "val_every_n_iters": 1000,
    "save_every_n_iters": 5000,
    "log_every_n_iters": 100,
    # augmentation (Run 1 ile AYNI)
    "augment": True,
    "aug_flip_p": 0.5,
    "aug_max_rot_deg": 3.0,
    "aug_max_translate_px": 20,
    "aug_morph_p": 0.5,
    "aug_boundary_noise_p": 0.3,
}

print(f"Config: {json.dumps(CONFIG, indent=2)}")


# %% --- CELL 3: Augmentation (Run 1 ile AYNI) ---

def random_hflip(img, p):
    if np.random.rand() < p:
        return cv2.flip(img, 1)
    return img

def random_rotate(img, max_deg):
    deg = np.random.uniform(-max_deg, max_deg)
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), deg, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_NEAREST,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=0)

def random_translate(img, max_px):
    tx = np.random.randint(-max_px, max_px + 1)
    ty = np.random.randint(-max_px, max_px + 1)
    M = np.float32([[1, 0, tx], [0, 1, ty]])
    h, w = img.shape[:2]
    return cv2.warpAffine(img, M, (w, h),
                          borderMode=cv2.BORDER_CONSTANT, borderValue=0)

def random_morphology(img, p):
    if np.random.rand() > p:
        return img
    k = np.random.choice([2, 3])
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    if np.random.rand() < 0.5:
        return cv2.erode(img, kernel, iterations=1)
    return cv2.dilate(img, kernel, iterations=1)

def random_boundary_noise(img, p, n_blobs=(1, 4), blob_px=(2, 5)):
    if np.random.rand() > p:
        return img
    edges = cv2.Canny(img, 50, 150)
    ys, xs = np.nonzero(edges)
    if len(ys) == 0:
        return img
    img = img.copy()
    for _ in range(np.random.randint(*n_blobs)):
        i = np.random.randint(len(ys))
        r = np.random.randint(*blob_px)
        val = 255 if np.random.rand() < 0.5 else 0
        cv2.circle(img, (int(xs[i]), int(ys[i])), r, int(val), -1)
    return img

def augment_view(img_u8):
    img_u8 = random_hflip(img_u8, CONFIG["aug_flip_p"])
    img_u8 = random_rotate(img_u8, CONFIG["aug_max_rot_deg"])
    img_u8 = random_translate(img_u8, CONFIG["aug_max_translate_px"])
    img_u8 = random_morphology(img_u8, CONFIG["aug_morph_p"])
    img_u8 = random_boundary_noise(img_u8, CONFIG["aug_boundary_noise_p"])
    return img_u8


# %% --- CELL 4: Dataset (DEGISTI: z-score yok, /100 olcekleme) ---

class BodyMDataset(Dataset):

    def __init__(self, data_dir, split="train", subject_ids=None, augment=False):
        self.data_dir = Path(data_dir)
        self.split = split
        self.augment = augment

        metadata = pd.read_csv(self.data_dir / "hwg_metadata.csv")
        measurements = pd.read_csv(self.data_dir / "measurements.csv")
        photo_map = pd.read_csv(self.data_dir / "subject_to_photo_map.csv")

        self.data = photo_map.merge(metadata, on="subject_id")
        self.data = self.data.merge(measurements, on="subject_id")

        if subject_ids is not None:
            self.data = self.data[
                self.data["subject_id"].isin(subject_ids)
            ].reset_index(drop=True)

        self.mask_dir = self.data_dir / "mask"
        self.mask_left_dir = self.data_dir / "mask_left"

        valid = []
        for idx in range(len(self.data)):
            pid = self.data.iloc[idx]["photo_id"]
            if (self.mask_dir / f"{pid}.png").exists() and \
               (self.mask_left_dir / f"{pid}.png").exists():
                valid.append(idx)
        self.data = self.data.iloc[valid].reset_index(drop=True)
        print(f"[{split} | augment={augment}] {len(self.data)} valid samples")

    def _load_view(self, path):
        img = np.array(Image.open(path).convert("L"))
        img = np.where(img > 127, 255, 0).astype(np.uint8)
        img = cv2.resize(img, (CONFIG["img_width"], CONFIG["img_height"]),
                         interpolation=cv2.INTER_NEAREST)
        return img

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        pid = row["photo_id"]

        frontal = self._load_view(self.mask_dir / f"{pid}.png")
        lateral = self._load_view(self.mask_left_dir / f"{pid}.png")

        if self.augment:
            frontal = augment_view(frontal)
            lateral = augment_view(lateral)

        combined = np.concatenate(
            [frontal.astype(np.float32) / 255.0,
             lateral.astype(np.float32) / 255.0], axis=1)

        height_ch = np.full_like(combined, row["height_cm"] / 200.0)
        weight_ch = np.full_like(combined, row["weight_kg"] / 150.0)

        input_tensor = torch.from_numpy(
            np.stack([combined, height_ch, weight_ch], axis=0))

        # DEGISTI: z-score yerine eski /100 olcekleme
        target = torch.FloatTensor([row[m] / 100.0 for m in MEASUREMENT_NAMES])

        return input_tensor, target


# %% --- CELL 5: Model (AYNI) ---

class BMnet(nn.Module):
    """MNASNet1.0 backbone -> AdaptiveAvgPool -> MLP(128) -> 14 outputs"""

    def __init__(self, num_measurements=14, pretrained=True):
        super().__init__()
        weights = models.MNASNet1_0_Weights.DEFAULT if pretrained else None
        base = models.mnasnet1_0(weights=weights)
        self.features = base.layers
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Linear(1280, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_measurements)
        )
        self._init_head()

    def _init_head(self):
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.head(x)


# %% --- CELL 6: Logger (AYNI) ---

class TrainingLogger:

    def __init__(self, log_dir):
        self.log_dir = Path(log_dir)
        self.log_file = self.log_dir / "training_log.json"
        self.history = {
            "iter_losses": [],
            "val_checkpoints": [],
            "best_val_mae": float("inf"),
            "best_iter": 0,
            "current_iter": 0,
        }
        if self.log_file.exists():
            with open(self.log_file) as f:
                self.history.update(json.load(f))
            print(f"Loaded log: iter {self.history['current_iter']}")

    def log_iter(self, iteration, loss):
        self.history["iter_losses"].append((iteration, loss))
        self.history["current_iter"] = iteration
        if len(self.history["iter_losses"]) > 10000:
            self.history["iter_losses"] = self.history["iter_losses"][-10000:]

    def log_val(self, iteration, val_loss, mae, tp50, tp90):
        self.history["val_checkpoints"].append(
            (iteration, val_loss, mae, tp50, tp90))
        is_best = mae < self.history["best_val_mae"]
        if is_best:
            self.history["best_val_mae"] = mae
            self.history["best_iter"] = iteration
        return is_best

    def save(self):
        with open(self.log_file, "w") as f:
            json.dump(self.history, f, indent=2)


# %% --- CELL 7: Metrics & Checkpointing (DEGISTI: denormalizasyon yok) ---

def compute_metrics(preds, targets):
    """Pred/target /100 olcekli gelir; *1000 = mm."""
    abs_errors = torch.abs(preds - targets) * 1000.0

    return {
        "mae_mm": abs_errors.mean().item(),
        "tp50_mm": torch.quantile(abs_errors, 0.50, dim=0).mean().item(),
        "tp75_mm": torch.quantile(abs_errors, 0.75, dim=0).mean().item(),
        "tp90_mm": torch.quantile(abs_errors, 0.90, dim=0).mean().item(),
        "mae_per_measurement": {
            name: abs_errors[:, i].mean().item()
            for i, name in enumerate(MEASUREMENT_NAMES)
        },
    }

def save_checkpoint(model, optimizer, scheduler, iteration, logger, path, is_best=False):
    state = {
        "iteration": iteration,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_mae": logger.history["best_val_mae"],
        "config": CONFIG,
    }
    torch.save(state, path)
    if is_best:
        best_path = Path(path).parent / "best_model.pth"
        torch.save(state, best_path)
        print(f"  *** BEST MODEL saved: {best_path}")

def load_checkpoint(model, optimizer, scheduler, checkpoint_dir):
    path = Path(checkpoint_dir) / "latest_checkpoint.pth"
    if not path.exists():
        print("No checkpoint found, starting from scratch.")
        return 0
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if "scheduler_state_dict" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    start = ckpt["iteration"]
    print(f"Checkpoint loaded: iter {start}, "
          f"best val MAE: {ckpt.get('best_val_mae', float('inf')):.2f} mm")
    return start


# %% --- CELL 8: Split persistence (DEGISTI: norm_stats fonksiyonu silindi) ---

def get_or_create_split(train_dir):
    split_file = Path(CHECKPOINT_DIR) / "split.json"
    if split_file.exists():
        with open(split_file) as f:
            s = json.load(f)
        print(f"Split loaded: {len(s['train_ids'])} train / {len(s['val_ids'])} val subjects")
        return set(s["train_ids"]), set(s["val_ids"])

    full = BodyMDataset(train_dir, split="full", augment=False)
    all_subjects = full.data["subject_id"].unique()
    np.random.shuffle(all_subjects)
    n_val = max(1, int(len(all_subjects) * CONFIG["val_split"]))
    val_ids = all_subjects[:n_val].tolist()
    train_ids = all_subjects[n_val:].tolist()

    with open(split_file, "w") as f:
        json.dump({"train_ids": train_ids, "val_ids": val_ids}, f)
    print(f"Split created & saved: {len(train_ids)} train / {len(val_ids)} val subjects")
    return set(train_ids), set(val_ids)


# %% --- CELL 9: Validation (DEGISTI: norm_stats parametresi yok) ---

@torch.no_grad()
def validate(model, val_loader, criterion):
    model.eval()
    total_loss = 0
    all_preds, all_targets = [], []

    for inputs, targets in val_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        outputs = model(inputs)
        total_loss += criterion(outputs, targets).item()
        all_preds.append(outputs.cpu())
        all_targets.append(targets.cpu())

    avg_loss = total_loss / len(val_loader)
    metrics = compute_metrics(torch.cat(all_preds), torch.cat(all_targets))
    model.train()
    return avg_loss, metrics


# %% --- CELL 10: Training Loop (DEGISTI: norm_stats satirlari yok) ---

def train(resume=True):
    print("=" * 60)
    print("BMnet RUN 2 (augmentation only, no z-score)")
    print("=" * 60)

    train_dir = os.path.join(DATA_ROOT, "train")

    train_ids, val_ids = get_or_create_split(train_dir)

    train_dataset = BodyMDataset(train_dir, split="train",
                                 subject_ids=train_ids,
                                 augment=CONFIG["augment"])
    val_dataset = BodyMDataset(train_dir, split="val",
                               subject_ids=val_ids,
                               augment=False)

    train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"],
                              shuffle=True, num_workers=CONFIG["num_workers"],
                              pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"],
                            shuffle=False, num_workers=CONFIG["num_workers"],
                            pin_memory=True)

    print(f"\nTrain: {len(train_dataset)} samples, {len(train_loader)} iters/epoch")
    print(f"Val: {len(val_dataset)} samples")
    print(f"Total iterations: {CONFIG['total_iterations']}")

    model = BMnet(num_measurements=CONFIG["num_measurements"]).to(DEVICE)
    criterion = nn.L1Loss()
    optimizer = optim.Adam(model.parameters(), lr=CONFIG["lr"])

    milestones = [int(CONFIG["total_iterations"] * 0.75),
                  int(CONFIG["total_iterations"] * 0.88)]
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=milestones, gamma=CONFIG["lr_gamma"])

    logger = TrainingLogger(CHECKPOINT_DIR)
    start_iter = 0
    if resume:
        start_iter = load_checkpoint(model, optimizer, scheduler, CHECKPOINT_DIR)
        for param_group in optimizer.param_groups:
            if start_iter < milestones[0]:
                param_group['lr'] = CONFIG["lr"]
            elif start_iter < milestones[1]:
                param_group['lr'] = CONFIG["lr"] * CONFIG["lr_gamma"]
            else:
                param_group['lr'] = CONFIG["lr"] * (CONFIG["lr_gamma"] ** 2)

    print(f"\nTraining: iter {start_iter} -> {CONFIG['total_iterations']}")
    print(f"LR: {optimizer.param_groups[0]['lr']:.1e}")
    print("-" * 60)

    model.train()
    global_iter = start_iter
    data_iter = iter(train_loader)
    running_loss, running_count = 0, 0
    t_start = time.time()

    while global_iter < CONFIG["total_iterations"]:
        try:
            inputs, targets = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            inputs, targets = next(data_iter)

        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

        optimizer.zero_grad()
        loss = criterion(model(inputs), targets)
        loss.backward()
        optimizer.step()
        scheduler.step()

        global_iter += 1
        running_loss += loss.item()
        running_count += 1

        if global_iter % CONFIG["log_every_n_iters"] == 0:
            avg = running_loss / running_count
            elapsed = time.time() - t_start
            ips = CONFIG["log_every_n_iters"] / elapsed
            remaining = (CONFIG["total_iterations"] - global_iter) / ips / 3600
            print(f"Iter {global_iter:6d}/{CONFIG['total_iterations']} | "
                  f"Loss: {avg:.4f} | "
                  f"LR: {optimizer.param_groups[0]['lr']:.1e} | "
                  f"{ips:.1f} it/s | ~{remaining:.1f}h left")
            logger.log_iter(global_iter, avg)
            running_loss, running_count = 0, 0
            t_start = time.time()

        if global_iter % CONFIG["val_every_n_iters"] == 0:
            val_loss, metrics = validate(model, val_loader, criterion)
            is_best = logger.log_val(global_iter, val_loss,
                                     metrics["mae_mm"], metrics["tp50_mm"],
                                     metrics["tp90_mm"])
            print(f"\n  [VAL iter {global_iter}] "
                  f"Loss: {val_loss:.4f} | "
                  f"MAE: {metrics['mae_mm']:.1f}mm | "
                  f"TP50: {metrics['tp50_mm']:.1f}mm | "
                  f"TP90: {metrics['tp90_mm']:.1f}mm | "
                  f"{'*** BEST ***' if is_best else ''}")
            if is_best:
                save_checkpoint(model, optimizer, scheduler, global_iter, logger,
                                os.path.join(CHECKPOINT_DIR, "latest_checkpoint.pth"),
                                is_best=True)
            model.train()

        if global_iter % CONFIG["save_every_n_iters"] == 0:
            save_checkpoint(model, optimizer, scheduler, global_iter, logger,
                            os.path.join(CHECKPOINT_DIR, "latest_checkpoint.pth"))
            logger.save()

    save_checkpoint(model, optimizer, scheduler, global_iter, logger,
                    os.path.join(CHECKPOINT_DIR, "latest_checkpoint.pth"))
    logger.save()

    print("\n" + "=" * 60)
    print(f"Training Complete! Best: iter {logger.history['best_iter']}, "
          f"val MAE: {logger.history['best_val_mae']:.2f} mm")
    print("=" * 60)
    return model, logger


# %% --- CELL 11: Test Evaluation (DEGISTI: norm_stats yok) ---

@torch.no_grad()
def evaluate_test(model_path, test_dir, split_name="TestA"):
    print(f"\n{'='*60}\n{split_name} Evaluation\n{'='*60}")

    model = BMnet(num_measurements=CONFIG["num_measurements"]).to(DEVICE)
    ckpt = torch.load(model_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    dataset = BodyMDataset(test_dir, split=split_name, augment=False)
    loader = DataLoader(dataset, batch_size=CONFIG["batch_size"],
                        shuffle=False, num_workers=CONFIG["num_workers"])

    all_preds, all_targets = [], []
    for inputs, targets in tqdm(loader, desc=split_name):
        all_preds.append(model(inputs.to(DEVICE)).cpu())
        all_targets.append(targets)

    metrics = compute_metrics(torch.cat(all_preds), torch.cat(all_targets))

    print(f"\nOverall (mm):")
    print(f"  MAE:  {metrics['mae_mm']:.2f}")
    print(f"  TP50: {metrics['tp50_mm']:.2f}")
    print(f"  TP75: {metrics['tp75_mm']:.2f}")
    print(f"  TP90: {metrics['tp90_mm']:.2f}")
    print(f"\nPer-Measurement MAE (mm):")
    for name, mae in metrics["mae_per_measurement"].items():
        print(f"  {name:25s}: {mae:.2f}")
    return metrics


# %% --- CELL 12: Training Curves (AYNI) ---

def plot_training(log_dir):
    log_file = Path(log_dir) / "training_log.json"
    if not log_file.exists():
        print("No log found.")
        return
    with open(log_file) as f:
        h = json.load(f)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    if h["iter_losses"]:
        iters, losses = zip(*h["iter_losses"])
        axes[0].plot(iters, losses, alpha=0.3, color="blue", linewidth=0.5)
        window = min(50, len(losses))
        if window > 1:
            ma = np.convolve(losses, np.ones(window) / window, mode='valid')
            axes[0].plot(iters[window - 1:], ma, color="red", linewidth=2)
        axes[0].set(xlabel="Iteration", ylabel="L1 Loss (/100 scale)",
                    title="Training Loss")
        axes[0].grid(True, alpha=0.3)

    if h["val_checkpoints"]:
        v_iters, v_loss, v_mae, v_tp50, v_tp90 = zip(*h["val_checkpoints"])
        axes[1].plot(v_iters, v_loss, 'o-', color="red", label="Val Loss")
        axes[1].set(xlabel="Iteration", ylabel="Val L1 Loss (/100 scale)",
                    title="Validation Loss")
        axes[1].legend(); axes[1].grid(True, alpha=0.3)

        axes[2].plot(v_iters, v_mae, 'o-', color="orange", label="MAE")
        axes[2].plot(v_iters, v_tp50, 's-', color="green", label="TP50")
        axes[2].plot(v_iters, v_tp90, '^-', color="red", label="TP90")
        axes[2].set(xlabel="Iteration", ylabel="mm", title="Validation Metrics (mm)")
        axes[2].legend(); axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(Path(log_dir) / "training_curves.png", dpi=150)
    plt.show()


# %% --- CELL 13: Inference (DEGISTI: denormalizasyon yok, *100) ---

class BMnetInference:

    def __init__(self, model_path, device=None):
        self.device = device or DEVICE
        self.model = BMnet(num_measurements=len(MEASUREMENT_NAMES),
                           pretrained=False)
        ckpt = torch.load(model_path, map_location=self.device, weights_only=False)
        self.model.load_state_dict(ckpt["model_state_dict"])
        self.model.to(self.device).eval()

    def _prep_silhouette(self, img):
        if isinstance(img, (str, Path)):
            img = Image.open(img).convert("L")
        else:
            img = img.convert("L")
        arr = np.array(img)
        arr = np.where(arr > 127, 255, 0).astype(np.uint8)
        arr = cv2.resize(arr, (CONFIG["img_width"], CONFIG["img_height"]),
                         interpolation=cv2.INTER_NEAREST)
        return arr.astype(np.float32) / 255.0

    @torch.no_grad()
    def predict(self, frontal, lateral, height_cm, weight_kg):
        f = self._prep_silhouette(frontal)
        l = self._prep_silhouette(lateral)

        combined = np.concatenate([f, l], axis=1)
        h_ch = np.full_like(combined, height_cm / 200.0)
        w_ch = np.full_like(combined, weight_kg / 150.0)

        tensor = torch.from_numpy(
            np.stack([combined, h_ch, w_ch], axis=0)
        ).unsqueeze(0).to(self.device)

        out = self.model(tensor).cpu().numpy()[0] * 100.0  # /100 -> cm

        return {name: round(float(v), 2)
                for name, v in zip(MEASUREMENT_NAMES, out)}


# %% --- CELL 14: RUN ---

if __name__ == "__main__":
    model, logger = train(resume=True)
    plot_training(CHECKPOINT_DIR)

    best_path = os.path.join(CHECKPOINT_DIR, "best_model.pth")

    test_a = os.path.join(DATA_ROOT, "testA")
    if os.path.exists(test_a):
        evaluate_test(best_path, test_a, "TestA")

    test_b = os.path.join(DATA_ROOT, "testB")
    if os.path.exists(test_b):
        evaluate_test(best_path, test_b, "TestB")

In [ ]:
ds = BodyMDataset(os.path.join(DATA_ROOT, "train"), split="check", augment=True)
fig, axes = plt.subplots(2, 4, figsize=(14, 8))
for j in range(4):
    img, h, w, t = ds[j]
    arr = img.numpy()
    axes[0][j].imshow(arr[:, :CONFIG["img_width"]], cmap="gray")
    axes[1][j].imshow(arr[:, CONFIG["img_width"]:], cmap="gray")
    for a in (axes[0][j], axes[1][j]): a.axis("off")
plt.suptitle("Augmentation kontrolu: delikler/tirtiklar var mi, vucut hala taninabilir mi?")
plt.show()

In [ ]:
tmp = BodyMDataset(os.path.join(DATA_ROOT, "train"), split="chk", augment=False)
dl = DataLoader(tmp, batch_size=4, num_workers=0)
img_u8, h, w, t = next(iter(dl))
x = img_u8.float().div(255.0).unsqueeze(1)
inputs = torch.cat([x, h.view(-1,1,1,1).expand_as(x), w.view(-1,1,1,1).expand_as(x)], dim=1)
print(inputs.shape, inputs.min().item(), inputs.max().item(), inputs.dtype)

In [ ]:
best_path = os.path.join("/content/drive/MyDrive/BodyM_Dev/bodymmodels/checkpoints_aug_only/best_model.pth")
evaluate_test(best_path, os.path.join(DATA_ROOT, "testA"), "TestA")
evaluate_test(best_path, os.path.join(DATA_ROOT, "testB"), "TestB")

In [ ]:
# Bayrakli train subject'lerini yeniden bul ve gorsellestir
from sklearn.linear_model import LinearRegression

train_dir = os.path.join(DATA_ROOT, "train")
tr_ds = BodyMDataset(train_dir, split="train", augment=False)
tr = tr_ds.data.drop_duplicates("subject_id").copy()
tr["bmi"] = tr.weight_kg / (tr.height_cm / 100) ** 2

def make_X(d):
    cols = [d.height_cm.values, d.weight_kg.values, d.bmi.values]
    if "gender" in d.columns:
        cols.append((d.gender.astype(str).str.lower()
                     .isin(["female", "f"])).astype(float).values)
    return np.stack(cols, axis=1)

train_flagged = {}
for m in ["chest", "waist", "hip"]:
    reg = LinearRegression().fit(make_X(tr), tr[m].values)
    z = (tr[m].values - reg.predict(make_X(tr))) / \
        (tr[m].values - reg.predict(make_X(tr))).std()
    for sid, zv in zip(tr.subject_id[np.abs(z) > 3], z[np.abs(z) > 3]):
        train_flagged.setdefault(sid, []).append(f"{m}:z={zv:.1f}")

print(f"Toplam bayrakli train subject: {len(train_flagged)}")

# Ilk 10'unu ciz (her birinden 1 foto yeter)
for sid, reasons in list(train_flagged.items())[:10]:
    pid = tr_ds.data[tr_ds.data.subject_id == sid].photo_id.iloc[0]
    row = tr[tr.subject_id == sid].iloc[0]
    fig, ax = plt.subplots(1, 2, figsize=(6, 4))
    ax[0].imshow(tr_ds._load_view(tr_ds.mask_dir / f"{pid}.png"), cmap="gray")
    ax[1].imshow(tr_ds._load_view(tr_ds.mask_left_dir / f"{pid}.png"), cmap="gray")
    for a in ax: a.axis("off")
    fig.suptitle(f"{sid[:10]} | {row.height_cm:.0f}cm {row.weight_kg:.0f}kg "
                 f"BMI={row.bmi:.1f} | chest={row.chest:.0f} "
                 f"waist={row.waist:.0f} hip={row.hip:.0f}\n{reasons}",
                 fontsize=9)
    plt.show()